# A2A Client - Cat Shop Assistant

This notebook demonstrates how to use the **A2A (Agent-to-Agent) protocol** to communicate with a remote agent.

We'll connect to a Cat Shop Assistant agent running on `localhost:9999`, resolve its Agent Card, and send it a series of messages.

> **Prerequisites:** Make sure the A2A server is running (`uv run a2a/server.py`) before executing this notebook.

## Imports & Configuration

In [1]:
import uuid

import httpx

from a2a.types import (
    Message,
    MessageSendParams,
    Part,
    Role,
    SendMessageRequest,
    SendMessageSuccessResponse,
    Task,
    TextPart,
)

from a2a.client import A2ACardResolver, A2AClient

AGENT_URL = "http://localhost:9999/"

## Helper: Extract Text from Response

A2A responses can be either a `Message` or a `Task`. This helper extracts the text content from either type.

In [2]:
def extract_text(response) -> str:
    """Pull the agent's text out of a SendMessageResponse."""
    root = response.root

    if not isinstance(root, SendMessageSuccessResponse):
        return f"[Error] {root.error.message}"

    result = root.result

    if isinstance(result, Message):
        return " ".join(p.root.text for p in result.parts if isinstance(p.root, TextPart))

    if isinstance(result, Task):
        # Check status message first, then artifacts
        if result.status and result.status.message:
            return " ".join(p.root.text for p in result.status.message.parts if isinstance(p.root, TextPart))
        if result.artifacts:
            texts = []
            for artifact in result.artifacts:
                texts.extend(p.root.text for p in artifact.parts if isinstance(p.root, TextPart))
            return " ".join(texts)

    return f"[Unexpected response: {type(result).__name__}]"

## Resolve the Agent Card

Every A2A agent exposes an **Agent Card** at `/.well-known/agent-card.json`. This card describes the agent's name, version, capabilities, and skills.

In [3]:
httpx_client = httpx.AsyncClient()

resolver = A2ACardResolver(httpx_client=httpx_client, base_url=AGENT_URL)
agent_card = await resolver.get_agent_card()

print(f"Connected to: {agent_card.name} (v{agent_card.version})")
print(f"Skills: {', '.join(s.name for s in agent_card.skills)}")

Connected to: Cat Shop Assistant (v1.0.0)
Skills: Product Recommendations


## Create the A2A Client

In [4]:
client = A2AClient(httpx_client=httpx_client, agent_card=agent_card)

/var/folders/15/gwnn0_md7n967tk5k2f7y84r0000gn/T/ipykernel_25354/4224200481.py:1: DeprecationWarning: A2AClient is deprecated and will be removed in a future version. Use ClientFactory to create a client with a JSON-RPC transport.
  client = A2AClient(httpx_client=httpx_client, agent_card=agent_card)


## Send Messages to the Agent

We'll send a series of queries and print the agent's responses. Each message is wrapped in a `SendMessageRequest` with a unique ID.

In [5]:
queries = [
    "Hello!",
    "My cat needs a new toy",
    "What about some treats?",
    "Surprise me!",
]

for query in queries:
    print(f"You: {query}")
    request = SendMessageRequest(
        id=uuid.uuid4().hex,
        params=MessageSendParams(
            message=Message(
                role=Role.user,
                message_id=uuid.uuid4().hex,
                parts=[Part(root=TextPart(text=query))],
            ),
        ),
    )
    response = await client.send_message(request)
    print(f"Agent: {extract_text(response)}\n")

You: Hello!
Agent: Hello! Welcome to the Cat Shop. How can I assist you in finding the perfect products for your feline friend today?

You: My cat needs a new toy
Agent: We have some great toys for your cat! Here are a few options:
- Feather Wand for $8.99
- Laser Pointer for $12.99
- Catnip Mouse for $4.99

Would you like to see more details about any of these, or do you have a particular preference?

You: What about some treats?
Agent: We have some tasty treats for your cat! For example, "Salmon Crunchies" for $6.49 and "Tuna Treats" for $5.99. Would you like to know more about these or see other options?

You: Surprise me!
Agent: How about a fun toy for your cat? We have a Feather Wand, a Laser Pointer, and a Catnip Mouse. Would you like to hear more about any of these?



In [6]:
await httpx_client.aclose()